# Race Against the Machine (TII) + Spyx

This notebook is for the TII Race Against the Machine dataset.

Status in Tonic:
- No direct dataset class discovered in `tonic.datasets` here.

Goal:
- Provide a clean adapter path for local dataset exports.
- Benchmark multiple Spyx architectures with shared preprocessing.

In [ ]:
from pathlib import Path

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

import spyx.models as fm

DATA_ROOT = Path("../data/RaceAgainstMachine_TII")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT.resolve())


def tonic_has_race_tii():
    import tonic.datasets as d
    return any(("race" in n.lower() and "machine" in n.lower()) or "tii" in n.lower() for n in dir(d))

print("tonic_has_race_tii =", tonic_has_race_tii())
print("expected local artifacts: events.{h5,npz}, imu.csv, pose.csv")

In [ ]:
def synthetic_race_batch(batch=4, T=24, H=72, W=128, C=2, seed=2):
    rng = np.random.default_rng(seed)
    x = rng.poisson(0.025, size=(T, batch, H, W, C)).astype(np.float32)
    imu = rng.normal(size=(batch, 6)).astype(np.float32)
    y = rng.normal(size=(batch, 6)).astype(np.float32)
    return jnp.asarray(x), jnp.asarray(imu), jnp.asarray(y)


def race_models(input_hw=(72, 128), input_channels=2):
    base_cfg = fm.ConvConfig(input_hw=input_hw, input_channels=input_channels, channels1=24, channels2=48, output_dim=64)
    imu_cfg = fm.IMUConditionedConfig(vision_cfg=base_cfg, imu_dim=6, fused_dim=96, output_dim=6)
    rec_cfg = fm.VisualIMURecurrentConfig(vision_cfg=base_cfg, imu_dim=6, hidden_dim=96, output_dim=6)
    kf_cfg = fm.KalmanFusionConfig(vision_cfg=base_cfg, imu_dim=6, state_dim=32, output_dim=6)
    return {
        "imu_conditioned": lambda x, u: fm.IMUConditionedVisualSNN(imu_cfg)(x, u),
        "recurrent_fusion": lambda x, u: fm.VisualIMURecurrentFusionBlock(rec_cfg)(x, u),
        "kalman_surrogate": lambda x, u: fm.KalmanStyleSpikingFusionSurrogate(kf_cfg)(x, u),
    }


x, u, y = synthetic_race_batch()
for name, fn in race_models().items():
    net = hk.without_apply_rng(hk.transform(fn))
    params = net.init(jax.random.PRNGKey(0), x, u)
    pred, aux = net.apply(params, x, u)
    mse = jnp.mean((pred - y) ** 2)
    print(name, pred.shape, float(mse), "spike_rate", np.asarray(aux["spike_rate"]))

## Next Steps for Race Against the Machine (TII)

1. Wire real events/IMU/pose files into `../data/RaceAgainstMachine_TII`.
2. Add race-specific metrics (lap progression, gate consistency, drift).
3. Expand model bank with additional recurrent variants from Spyx fusion modules.